# Notebook 1 — Thu thập, hợp nhất và làm sạch dữ liệu báo chí tiếng Việt

## 1. Bối cảnh

Sự gia tăng nhanh của dữ liệu báo chí tiếng Việt trên các nền tảng số đặt ra nhu cầu tổ chức, khám phá và phân nhóm văn bản một cách tự động. Tuy nhiên, văn bản tiếng Việt có nhiều đặc thù như từ ghép đa âm tiết, tính đa nghĩa và mức độ chồng lấn chủ đề giữa các chuyên mục, khiến bài toán gom cụm văn bản trở nên khó hơn so với dữ liệu có cấu trúc.

## 2. Mục tiêu của notebook

Notebook này thực hiện bước chuẩn bị dữ liệu ban đầu cho bài toán **gom cụm văn bản báo chí tiếng Việt**.

Các mục tiêu chính gồm:

- Đọc và hợp nhất dữ liệu từ 6 chuyên mục báo chí.
- Kiểm tra chất lượng dữ liệu ban đầu.
- Phát hiện giá trị thiếu, URL trùng và nội dung trùng lặp.
- Tạo cột văn bản tổng hợp từ `title`, `description` và `content`.
- Lọc bỏ các bài viết quá ngắn hoặc không đủ thông tin.
- Lưu bộ dữ liệu sạch để sử dụng cho bước tiền xử lý văn bản tiếng Việt ở Notebook 2.

## 3. Lưu ý về bản chất bài toán

Đây là bài toán **gom cụm văn bản**, không phải bài toán phân loại văn bản.

Trong notebook này, cột `category_clean` chỉ được giữ lại để:

- kiểm tra phân bố dữ liệu,
- mô tả bộ dữ liệu,
- và có thể dùng làm nhãn tham khảo khi tính một số chỉ số đánh giá ngoài như ARI, NMI hoặc Purity.

Cột `category_clean` **không được sử dụng như nhãn huấn luyện** và không dùng để xây dựng mô hình phân loại.

## 4. Đầu ra mong đợi

Sau notebook này, dữ liệu sạch sẽ được lưu thành file:

`data/processed/news_clean.csv`

Ngoài ra, notebook cũng xuất một số bảng thống kê phục vụ báo cáo/paper:

- `outputs/reports/dataset_statistics.csv`
- `outputs/reports/category_distribution.csv`

File `news_clean.csv` là đầu vào chính cho Notebook 02 — Vietnamese Text Preprocessing.

In [1]:
# Cell 2: Import thư viện và khai báo đường dẫn dữ liệu
# Cell này chuẩn hóa cấu trúc thư mục để các notebook sau đọc/ghi dữ liệu nhất quán.

import pandas as pd
import numpy as np
from pathlib import Path

# Thư mục gốc của project
PROJECT_ROOT = Path(r"D:\PROJECT")

# Thư mục dữ liệu
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

# Thư mục lưu kết quả
OUTPUT_DIR = PROJECT_ROOT / "outputs"
REPORT_DIR = OUTPUT_DIR / "reports"

# Tạo thư mục output nếu chưa có
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Danh sách file dữ liệu gốc
FILES = {
    "cong_nghe": "cong_nghe.csv",
    "du_lich": "du_lich.csv",
    "giao_duc": "giao_duc.csv",
    "suc_khoe": "suc_khoe.csv",
    "the_thao": "the_thao.csv",
    "xe": "xe.csv",
}

# File output chính của Notebook 1
OUTPUT_CLEAN = PROCESSED_DIR / "news_clean.csv"
OUTPUT_CATEGORY_REPORT = REPORT_DIR / "category_distribution.csv"
OUTPUT_DATASET_STATS = REPORT_DIR / "dataset_statistics.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DIR:", RAW_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("REPORT_DIR:", REPORT_DIR)

print("\nDanh sách file đầu vào:")
for category, file_name in FILES.items():
    file_path = RAW_DIR / file_name
    print("-", category, "|", file_path, "| tồn tại:", file_path.exists())

print("\nFile dữ liệu sạch sẽ lưu tại:")
print(OUTPUT_CLEAN)

PROJECT_ROOT: D:\PROJECT
RAW_DIR: D:\PROJECT\data\raw
PROCESSED_DIR: D:\PROJECT\data\processed
REPORT_DIR: D:\PROJECT\outputs\reports

Danh sách file đầu vào:
- cong_nghe | D:\PROJECT\data\raw\cong_nghe.csv | tồn tại: True
- du_lich | D:\PROJECT\data\raw\du_lich.csv | tồn tại: True
- giao_duc | D:\PROJECT\data\raw\giao_duc.csv | tồn tại: True
- suc_khoe | D:\PROJECT\data\raw\suc_khoe.csv | tồn tại: True
- the_thao | D:\PROJECT\data\raw\the_thao.csv | tồn tại: True
- xe | D:\PROJECT\data\raw\xe.csv | tồn tại: True

File dữ liệu sạch sẽ lưu tại:
D:\PROJECT\data\processed\news_clean.csv


In [2]:
# Cell 3: Đọc và gộp 6 file dữ liệu thành một DataFrame duy nhất
# Mỗi file tương ứng với một chuyên mục báo chí.

df_list = []

for category, file_name in FILES.items():
    file_path = RAW_DIR / file_name
    
    # Đọc từng file CSV
    df_temp = pd.read_csv(file_path).copy()
    
    # Gắn nhãn chuyên mục theo tên file để đảm bảo nhất quán
    df_temp["category_clean"] = category
    df_temp["source_file"] = file_name
    
    # Lưu vào danh sách
    df_list.append(df_temp)

# Gộp tất cả file thành một bảng
df_main = pd.concat(df_list, ignore_index=True)

# Tạo doc_id tạm thời, sau khi làm sạch sẽ reset lại doc_id lần nữa
df_main.insert(0, "doc_id", range(1, len(df_main) + 1))

print("Kích thước dữ liệu sau khi gộp:", df_main.shape)

print("\nCác cột hiện có:")
print(df_main.columns.tolist())

print("\nSố bài theo chuyên mục:")
print(df_main["category_clean"].value_counts())

print("\nSố bài theo file nguồn:")
print(df_main["source_file"].value_counts())

display(df_main.head(3))

Kích thước dữ liệu sau khi gộp: (8830, 9)

Các cột hiện có:
['doc_id', 'source', 'category_clean', 'title', 'description', 'content', 'published_date', 'url', 'source_file']

Số bài theo chuyên mục:
category_clean
du_lich      1984
giao_duc     1983
suc_khoe     1976
the_thao     1407
cong_nghe     991
xe            489
Name: count, dtype: int64

Số bài theo file nguồn:
source_file
du_lich.csv      1984
giao_duc.csv     1983
suc_khoe.csv     1976
the_thao.csv     1407
cong_nghe.csv     991
xe.csv            489
Name: count, dtype: int64


,doc_id,source,category_clean,title,description,content,published_date,url,source_file
0,1,tuoitre,cong_nghe,Làn sóng máy tính lượng tử sau cơn sốt AI,"Quý 1-2026, thế giới chứng kiến loạt công ty v...","Đáng chú ý, khác với các giai đoạn sàng lọc cô...",2026-04-11T11:36:52+07:00,https://tuoitre.vn/lan-song-may-tinh-luong-tu-...,cong_nghe.csv
1,2,tuoitre,cong_nghe,Meta đưa AI vào Messenger trả lời khách hàng 24/7,Khách hàng nhắn tin lúc nửa đêm vẫn có người t...,Meta vừa giới thiệu trợ lý kinh doanh Business...,2026-04-11T07:11:28+07:00,https://tuoitre.vn/meta-dua-ai-vao-messenger-t...,cong_nghe.csv
2,3,tuoitre,cong_nghe,Hà Tiên sử dụng trí tuệ nhân tạo phục vụ hành ...,"Ngày 10-4, UBND phường Hà Tiên (An Giang) ra m...","Phát biểu tại buổi lễ ra mắt, ông Trần Hải Quố...",2026-04-10T16:23:50+07:00,https://tuoitre.vn/ha-tien-su-dung-tri-tue-nha...,cong_nghe.csv


In [3]:
# Cell 4: Kiểm tra nhanh chất lượng dữ liệu sau khi gộp
# Cell này giúp phát hiện thiếu dữ liệu, trùng URL và trùng nội dung trước khi làm sạch sâu hơn.

print("Kích thước dữ liệu hiện tại:", df_main.shape)

print("\nSố giá trị thiếu theo cột:")
display(df_main.isna().sum().to_frame("missing_count"))

# Kiểm tra trùng URL nếu có cột url
if "url" in df_main.columns:
    url_duplicated = df_main["url"].duplicated().sum()
    print("\nSố URL trùng:", url_duplicated)
else:
    url_duplicated = None
    print("\nKhông có cột url để kiểm tra trùng URL.")

# Kiểm tra trùng theo title + content
title_content_duplicated = df_main.duplicated(subset=["title", "content"]).sum()
print("Số bản ghi trùng theo cặp title + content:", title_content_duplicated)

print("\nPhân bố dữ liệu theo chuyên mục:")
category_count = df_main["category_clean"].value_counts().to_frame("n_docs")
category_count["ratio"] = (category_count["n_docs"] / len(df_main) * 100).round(2)
display(category_count)

print("\nKiểu dữ liệu của các cột:")
display(df_main.dtypes.to_frame("dtype"))

Kích thước dữ liệu hiện tại: (8830, 9)

Số giá trị thiếu theo cột:


,missing_count
doc_id,0
source,0
category_clean,0
title,0
description,0
content,0
published_date,0
url,0
source_file,0



Số URL trùng: 101
Số bản ghi trùng theo cặp title + content: 102

Phân bố dữ liệu theo chuyên mục:


,n_docs,ratio
category_clean,,
du_lich,1984,22.47
giao_duc,1983,22.46
suc_khoe,1976,22.38
the_thao,1407,15.93
cong_nghe,991,11.22
xe,489,5.54



Kiểu dữ liệu của các cột:


,dtype
doc_id,int64
source,object
category_clean,object
title,object
description,object
content,object
published_date,object
url,object
source_file,object


### Nhận xét nhanh sau bước kiểm tra dữ liệu

Dữ liệu sau khi gộp có quy mô 8.830 bài báo từ 6 chuyên mục. Các cột văn bản chính như `title`, `description` và `content` được kiểm tra để đảm bảo đủ dữ liệu cho bước tiền xử lý tiếp theo.

Kết quả kiểm tra cho thấy dữ liệu còn tồn tại một số bản ghi trùng, chủ yếu theo URL. Vì vậy, notebook tiếp tục thực hiện bước loại trùng theo `url` và theo cặp `title + content` nhằm tránh việc các bài giống nhau làm sai lệch kết quả gom cụm.

Cột `category_clean` chỉ được giữ lại để mô tả dữ liệu và đánh giá hậu nghiệm sau khi gom cụm, không được sử dụng làm đầu vào huấn luyện mô hình.

In [4]:
# Cell 6: tạo cột văn bản chính và các cột chuẩn hóa tối thiểu để phục vụ lọc trùng
# Cell này giúp chuẩn bị dữ liệu cho bước dedup và làm sạch bằng cách gom nội dung văn bản về một dạng nhất quán hơn.

df_work = df_main.copy()

df_work["raw_text"] = (
    df_work["title"].astype(str).str.strip() + " . " +
    df_work["description"].astype(str).str.strip() + " . " +
    df_work["content"].astype(str).str.strip()
)

for col in ["title", "description", "content", "raw_text"]:
    df_work[f"{col}_norm"] = (
        df_work[col]
        .astype(str)
        .str.lower()
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

print("Kích thước dữ liệu làm việc:", df_work.shape)

show_cols = [
    "doc_id", "category_clean", "title", "description", "content", "raw_text"
]
display(df_work[show_cols].head(2))

Kích thước dữ liệu làm việc: (8830, 14)


,doc_id,category_clean,title,description,content,raw_text
0,1,cong_nghe,Làn sóng máy tính lượng tử sau cơn sốt AI,"Quý 1-2026, thế giới chứng kiến loạt công ty v...","Đáng chú ý, khác với các giai đoạn sàng lọc cô...",Làn sóng máy tính lượng tử sau cơn sốt AI . Qu...
1,2,cong_nghe,Meta đưa AI vào Messenger trả lời khách hàng 24/7,Khách hàng nhắn tin lúc nửa đêm vẫn có người t...,Meta vừa giới thiệu trợ lý kinh doanh Business...,Meta đưa AI vào Messenger trả lời khách hàng 2...


In [5]:
# Cell 7: tính độ dài văn bản để phục vụ lọc bài quá ngắn và kiểm tra nhanh phân bố độ dài
# Cell này giúp xác định mức độ đầy đủ của nội dung trước khi loại trùng và làm sạch sâu hơn.

for col in ["title", "description", "content", "raw_text"]:
    df_work[f"{col}_word_count"] = (
        df_work[col]
        .astype(str)
        .str.split()
        .str.len()
    )

summary_cols = [
    "title_word_count",
    "description_word_count",
    "content_word_count",
    "raw_text_word_count",
]

print("Thống kê độ dài văn bản:")
display(df_work[summary_cols].describe().T)

print("\nSố bài có content dưới 30 từ:")
print((df_work["content_word_count"] < 30).sum())

Thống kê độ dài văn bản:


,count,mean,std,min,25%,50%,75%,max
title_word_count,8830.0,14.903737,3.655548,4.0,12.0,15.0,17.0,26.0
description_word_count,8830.0,37.753114,8.674203,15.0,31.0,37.0,44.0,59.0
content_word_count,8830.0,592.757191,302.067642,23.0,385.0,534.0,737.0,6774.0
raw_text_word_count,8830.0,647.414043,303.757898,82.0,437.0,589.0,795.0,6822.0



Số bài có content dưới 30 từ:
1


### Nhận xét nhanh về độ dài văn bản

Độ dài `content` trung bình khoảng **593 từ**, trung vị khoảng **534 từ**, cho thấy phần lớn bài viết có nội dung tương đối đầy đủ và giàu thông tin. Chỉ có **1 bài dưới 30 từ**, nên bước lọc bài quá ngắn nếu áp dụng sẽ gần như không làm thay đổi quy mô dữ liệu, nhưng vẫn nên giữ để đảm bảo tính nhất quán của pipeline.

In [6]:
# Cell 9: loại trùng theo URL và theo cặp title + content
# Cell này thực hiện bước làm sạch quan trọng để tránh nhiều bản ghi trùng gây nhiễu khi gom cụm.

before_rows = len(df_work)

df_dedup = df_work.drop_duplicates(subset=["url"], keep="first").copy()
after_url_dedup = len(df_dedup)

df_dedup = df_dedup.drop_duplicates(subset=["title_norm", "content_norm"], keep="first").copy()
after_title_content_dedup = len(df_dedup)

print("Số dòng ban đầu:", before_rows)
print("Sau khi loại trùng theo URL:", after_url_dedup)
print("Sau khi loại trùng theo title_norm + content_norm:", after_title_content_dedup)

print("\nSố dòng bị loại do URL trùng:", before_rows - after_url_dedup)
print("Số dòng bị loại thêm do title_norm + content_norm trùng:", after_url_dedup - after_title_content_dedup)

display(df_dedup[["doc_id", "category_clean", "title", "url"]].head(3))

Số dòng ban đầu: 8830
Sau khi loại trùng theo URL: 8729
Sau khi loại trùng theo title_norm + content_norm: 8728

Số dòng bị loại do URL trùng: 101
Số dòng bị loại thêm do title_norm + content_norm trùng: 1


,doc_id,category_clean,title,url
0,1,cong_nghe,Làn sóng máy tính lượng tử sau cơn sốt AI,https://tuoitre.vn/lan-song-may-tinh-luong-tu-...
1,2,cong_nghe,Meta đưa AI vào Messenger trả lời khách hàng 24/7,https://tuoitre.vn/meta-dua-ai-vao-messenger-t...
2,3,cong_nghe,Hà Tiên sử dụng trí tuệ nhân tạo phục vụ hành ...,https://tuoitre.vn/ha-tien-su-dung-tri-tue-nha...


### Nhận xét nhanh sau bước loại trùng

Sau bước loại trùng, bộ dữ liệu giảm từ **8.830** xuống **8.728** bài. Trong đó:
- **101 bài** bị loại do trùng URL,
- **1 bài** bị loại thêm do trùng theo cặp `title_norm + content_norm`.

Điều này cho thấy phần lớn vấn đề trùng lặp đến từ URL, còn mức độ trùng hoàn toàn theo tiêu đề và nội dung là rất thấp sau khi đã loại URL trùng. Bộ dữ liệu sau bước này đã sạch hơn và phù hợp hơn để chuyển sang bước tạo tập văn bản chính thức cho tiền xử lý.

In [7]:
# Cell 11: Lọc bài quá ngắn, reset doc_id và chốt file dữ liệu sạch
# File output của cell này là đầu vào chính cho Notebook 02.

MIN_WORDS = 30

df_final = df_dedup[df_dedup["content_word_count"] >= MIN_WORDS].copy()

# Reset lại index và doc_id sau khi loại trùng/lọc ngắn
df_final = df_final.reset_index(drop=True)
df_final["doc_id"] = np.arange(1, len(df_final) + 1)

print("--- TỔNG KẾT NOTEBOOK 1 ---")
print("Số lượng bài báo ban đầu:", len(df_main))
print("Số lượng bài báo sau loại trùng:", len(df_dedup))
print("Số lượng bài báo sau lọc ngắn:", len(df_final))
print("Tổng cộng đã loại bỏ:", len(df_main) - len(df_final), "bài")

print("\n--- PHÂN BỐ CHUYÊN MỤC CUỐI CÙNG ---")
category_final = df_final["category_clean"].value_counts().to_frame("n_docs")
category_final["percentage"] = (category_final["n_docs"] / len(df_final) * 100).round(2)
display(category_final)

# Bảng thống kê tổng quan để phục vụ báo cáo/paper
dataset_stats = pd.DataFrame({
    "metric": [
        "n_raw_documents",
        "n_after_deduplication",
        "n_after_length_filtering",
        "n_removed_total",
        "n_categories"
    ],
    "value": [
        len(df_main),
        len(df_dedup),
        len(df_final),
        len(df_main) - len(df_final),
        df_final["category_clean"].nunique()
    ]
})

display(dataset_stats)

# Giữ các cột cần thiết cho Notebook 02
final_cols = [
    "doc_id",
    "source",
    "category_clean",
    "title",
    "description",
    "content",
    "published_date",
    "url",
    "raw_text",
    "content_word_count",
    "raw_text_word_count"
]

df_final_export = df_final[final_cols].copy()

# Đổi tên cột độ dài cho gọn
df_final_export = df_final_export.rename(columns={
    "raw_text_word_count": "word_count"
})

# Lưu file theo cấu trúc project mới
df_final_export.to_csv(OUTPUT_CLEAN, index=False, encoding="utf-8-sig")
category_final.to_csv(OUTPUT_CATEGORY_REPORT, encoding="utf-8-sig")
dataset_stats.to_csv(OUTPUT_DATASET_STATS, index=False, encoding="utf-8-sig")

print("\nKích thước file sạch cuối cùng:", df_final_export.shape)

print("\nĐã lưu file sạch tại:")
print(OUTPUT_CLEAN)

print("\nĐã lưu bảng phân bố chuyên mục tại:")
print(OUTPUT_CATEGORY_REPORT)

print("\nĐã lưu thống kê dữ liệu tại:")
print(OUTPUT_DATASET_STATS)

display(df_final_export.head(3))

--- TỔNG KẾT NOTEBOOK 1 ---
Số lượng bài báo ban đầu: 8830
Số lượng bài báo sau loại trùng: 8728
Số lượng bài báo sau lọc ngắn: 8727
Tổng cộng đã loại bỏ: 103 bài

--- PHÂN BỐ CHUYÊN MỤC CUỐI CÙNG ---


,n_docs,percentage
category_clean,,
du_lich,1979,22.68
giao_duc,1941,22.24
suc_khoe,1929,22.10
the_thao,1402,16.07
cong_nghe,991,11.36
xe,485,5.56


,metric,value
0,n_raw_documents,8830
1,n_after_deduplication,8728
2,n_after_length_filtering,8727
3,n_removed_total,103
4,n_categories,6



Kích thước file sạch cuối cùng: (8727, 11)

Đã lưu file sạch tại:
D:\PROJECT\data\processed\news_clean.csv

Đã lưu bảng phân bố chuyên mục tại:
D:\PROJECT\outputs\reports\category_distribution.csv

Đã lưu thống kê dữ liệu tại:
D:\PROJECT\outputs\reports\dataset_statistics.csv


,doc_id,source,category_clean,title,description,content,published_date,url,raw_text,content_word_count,word_count
0,1,tuoitre,cong_nghe,Làn sóng máy tính lượng tử sau cơn sốt AI,"Quý 1-2026, thế giới chứng kiến loạt công ty v...","Đáng chú ý, khác với các giai đoạn sàng lọc cô...",2026-04-11T11:36:52+07:00,https://tuoitre.vn/lan-song-may-tinh-luong-tu-...,Làn sóng máy tính lượng tử sau cơn sốt AI . Qu...,1036,1096
1,2,tuoitre,cong_nghe,Meta đưa AI vào Messenger trả lời khách hàng 24/7,Khách hàng nhắn tin lúc nửa đêm vẫn có người t...,Meta vừa giới thiệu trợ lý kinh doanh Business...,2026-04-11T07:11:28+07:00,https://tuoitre.vn/meta-dua-ai-vao-messenger-t...,Meta đưa AI vào Messenger trả lời khách hàng 2...,565,615
2,3,tuoitre,cong_nghe,Hà Tiên sử dụng trí tuệ nhân tạo phục vụ hành ...,"Ngày 10-4, UBND phường Hà Tiên (An Giang) ra m...","Phát biểu tại buổi lễ ra mắt, ông Trần Hải Quố...",2026-04-10T16:23:50+07:00,https://tuoitre.vn/ha-tien-su-dung-tri-tue-nha...,Hà Tiên sử dụng trí tuệ nhân tạo phục vụ hành ...,313,366
